# 05 Create Chunks

Σε αυτό το notebook δημιουργούνται τα κειμενικά chunks από τις καθαρισμένες οικονομικές εκθέσεις. Η διαδικασία κρατά metadata για εταιρεία, έτος, έγγραφο και θέση του αποσπάσματος, ώστε τα chunks να μπορούν να χρησιμοποιηθούν σε retrieval και αξιολόγηση.


In [ ]:
# Uncomment if needed
# !pip install -q langchain-text-splitters pyarrow tqdm

In [ ]:
from pathlib import Path
import json
import warnings
import re

import pandas as pd
from tqdm.auto import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = BASE_DIR / "configs"
OUTPUTS_DIR = BASE_DIR / "outputs"

CLEANED_DIR = INTERIM_DIR / "cleaned"
CHUNKS_DIR = PROCESSED_DIR / "chunks"

PARSE_MANIFEST_PATH = INTERIM_DIR / "docling_parse_manifest.csv"
CLEANING_MANIFEST_PATH = INTERIM_DIR / "markdown_cleaning_manifest.csv"

CHUNKS_CSV_PATH = CHUNKS_DIR / "financebench_chunks.csv"
CHUNKS_PARQUET_PATH = CHUNKS_DIR / "financebench_chunks.parquet"
CHUNKING_MANIFEST_PATH = CHUNKS_DIR / "chunking_manifest.csv"
CHUNKING_STATS_PATH = CHUNKS_DIR / "chunking_stats.json"

for p in [CHUNKS_DIR, OUTPUTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("CLEANED_DIR:", CLEANED_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR)

In [ ]:
parse_manifest_df = pd.read_csv(PARSE_MANIFEST_PATH)
cleaning_manifest_df = pd.read_csv(CLEANING_MANIFEST_PATH)

print("parse_manifest_df:", parse_manifest_df.shape)
print("cleaning_manifest_df:", cleaning_manifest_df.shape)

In [ ]:
successful_cleaning_df = cleaning_manifest_df[cleaning_manifest_df["status"] == "success"].copy().reset_index(drop=True)

print("Successful cleaned files:", len(successful_cleaning_df))
successful_cleaning_df.head(2)

In [ ]:
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200
MIN_CHUNK_CHARS = 200

chunking_config = {
    "chunker": "table_aware",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "min_chunk_chars": MIN_CHUNK_CHARS
}

chunking_config

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n# ",
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    keep_separator=True
)

print(text_splitter)

In [ ]:
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def normalize_whitespace(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def estimate_tokens(text: str) -> int:
    # rough approximation for analysis only
    return max(1, len(text) // 4)

def safe_doc_id_from_filename(filename: str) -> str:
    return Path(filename).stem

In [ ]:
doc_inventory = successful_cleaning_df.copy()

doc_inventory["cleaned_markdown_exists"] = doc_inventory["cleaned_markdown_path"].apply(lambda x: Path(x).exists())
doc_inventory["doc_id"] = doc_inventory["markdown_filename"].apply(safe_doc_id_from_filename)

print("Inventory shape:", doc_inventory.shape)
doc_inventory.head(2)

In [ ]:
doc_inventory = doc_inventory[doc_inventory["cleaned_markdown_exists"]].copy().reset_index(drop=True)

print("Docs available for chunking:", len(doc_inventory))
doc_inventory.head(2)

In [ ]:
sample_path = Path(doc_inventory.loc[0, "cleaned_markdown_path"]) if len(doc_inventory) > 0 else None

if sample_path:
    sample_text = read_text(sample_path)
    print("Sample file:", sample_path.name)
    print(sample_text[:3000])
else:
    print("No cleaned markdown files found.")

In [ ]:
if sample_path:
    sample_text = normalize_whitespace(read_text(sample_path))
    sample_chunks = text_splitter.split_text(sample_text)

    print("Number of sample chunks:", len(sample_chunks))
    print("\nFirst chunk preview:\n")
    print(sample_chunks[0][:2000])
else:
    print("No sample available.")

In [ ]:
def chunk_document(doc_id: str, markdown_filename: str, markdown_path: Path, company=None, doc_type=None, doc_period=None):
    raw_text = read_text(markdown_path)
    normalized_text = normalize_whitespace(raw_text)

    split_chunks = text_splitter.split_text(normalized_text)

    records = []
    for i, chunk_text in enumerate(split_chunks):
        chunk_text = chunk_text.strip()

        if len(chunk_text) < MIN_CHUNK_CHARS:
            continue

        chunk_id = f"{doc_id}_chunk_{i:04d}"

        record = {
            "chunk_id": chunk_id,
            "doc_id": doc_id,
            "doc_name": doc_id,
            "markdown_filename": markdown_filename,
            "source_path": str(markdown_path),
            "company": company,
            "doc_type": doc_type,
            "doc_period": doc_period,
            "chunk_index": i,
            "chunk_text": chunk_text,
            "char_count": len(chunk_text),
            "token_estimate": estimate_tokens(chunk_text),
            "starts_with_heading": bool(re.match(r"^\s*#+\s+", chunk_text)),
            "contains_table_pipe": "|" in chunk_text,
        }

        records.append(record)

    return records

In [ ]:
all_chunk_records = []
chunking_manifest_records = []

for _, row in tqdm(doc_inventory.iterrows(), total=len(doc_inventory), desc="Chunking documents"):
    markdown_filename = row["markdown_filename"]
    markdown_path = Path(row["cleaned_markdown_path"])
    doc_id = row["doc_id"]

    record = {
        "doc_id": doc_id,
        "markdown_filename": markdown_filename,
        "cleaned_markdown_path": str(markdown_path),
        "status": None,
        "error_message": None,
        "n_chunks": 0,
        "total_chars": None,
    }

    try:
        raw_text = read_text(markdown_path)
        chunk_records = chunk_document(
            doc_id=doc_id,
            markdown_filename=markdown_filename,
            markdown_path=markdown_path,
            company=row.get("company"),
            doc_type=row.get("doc_type"),
            doc_period=row.get("doc_period"),
        )

        all_chunk_records.extend(chunk_records)

        record["status"] = "success"
        record["n_chunks"] = len(chunk_records)
        record["total_chars"] = len(raw_text)

    except Exception as e:
        record["status"] = "error"
        record["error_message"] = str(e)

    chunking_manifest_records.append(record)

chunks_df = pd.DataFrame(all_chunk_records)
chunking_manifest_df = pd.DataFrame(chunking_manifest_records)

print("chunks_df shape:", chunks_df.shape)
print("chunking_manifest_df shape:", chunking_manifest_df.shape)

In [ ]:
chunking_manifest_df.head()

In [ ]:
chunks_df.head(3)

In [ ]:
summary = {
    "n_documents": int(chunking_manifest_df["doc_id"].nunique()) if not chunking_manifest_df.empty else 0,
    "successful_documents": int((chunking_manifest_df["status"] == "success").sum()) if not chunking_manifest_df.empty else 0,
    "failed_documents": int((chunking_manifest_df["status"] == "error").sum()) if not chunking_manifest_df.empty else 0,
    "total_chunks": len(chunks_df),
    "avg_chunks_per_document": float(chunks_df.groupby("doc_id").size().mean()) if not chunks_df.empty else 0,
    "avg_chunk_chars": float(chunks_df["char_count"].mean()) if not chunks_df.empty else 0,
    "avg_chunk_token_estimate": float(chunks_df["token_estimate"].mean()) if not chunks_df.empty else 0,
}

pd.DataFrame([summary])

In [ ]:
chunks_df["char_count"].describe()

In [ ]:
short_chunks_df = chunks_df[chunks_df["char_count"] < MIN_CHUNK_CHARS].copy()

print("Short chunks:", len(short_chunks_df))
short_chunks_df.head()

In [ ]:
per_doc_chunk_counts = (
    chunks_df.groupby("doc_id")
    .size()
    .reset_index(name="n_chunks")
    .sort_values("n_chunks", ascending=False)
)

per_doc_chunk_counts.head(15)

In [ ]:
sample_doc_id = chunks_df["doc_id"].iloc[0] if not chunks_df.empty else None

if sample_doc_id:
    sample_doc_chunks = chunks_df[chunks_df["doc_id"] == sample_doc_id].copy()
    print("Sample doc_id:", sample_doc_id)
    print("Number of chunks:", len(sample_doc_chunks))

    for _, row in sample_doc_chunks.head(3).iterrows():
        print("\n" + "=" * 100)
        print("chunk_id:", row["chunk_id"])
        print("chunk_index:", row["chunk_index"])
        print("char_count:", row["char_count"])
        print("-" * 100)
        print(row["chunk_text"][:2000])
else:
    print("No chunks available.")

In [ ]:
chunks_df.to_csv(CHUNKS_CSV_PATH, index=False)
print("Saved CSV to:", CHUNKS_CSV_PATH)

try:
    chunks_df.to_parquet(CHUNKS_PARQUET_PATH, index=False)
    print("Saved Parquet to:", CHUNKS_PARQUET_PATH)
except ImportError as e:
    print("Parquet save skipped.")
    print("Reason:", e)

In [ ]:
chunking_manifest_df.to_csv(CHUNKING_MANIFEST_PATH, index=False)
print("Saved manifest to:", CHUNKING_MANIFEST_PATH)

In [ ]:
with open(CHUNKING_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved stats JSON to:", CHUNKING_STATS_PATH)

In [ ]:
output_summary = pd.DataFrame([{
    "chunks_csv_path": str(CHUNKS_CSV_PATH),
    "chunks_parquet_path": str(CHUNKS_PARQUET_PATH),
    "chunking_manifest_path": str(CHUNKING_MANIFEST_PATH),
    "chunking_stats_path": str(CHUNKING_STATS_PATH),
    "n_chunks": len(chunks_df),
    "n_docs_chunked": chunking_manifest_df["doc_id"].nunique() if not chunking_manifest_df.empty else 0
}])

output_summary

## Συμπέρασμα

Σε αυτό το notebook:

- φορτώθηκαν τα cleaned markdown documents
- εφαρμόστηκε recursive chunking
- διατηρήθηκαν metadata ανά chunk
- αποθηκεύτηκαν chunks και chunking manifest
- δημιουργήθηκαν βασικά στατιστικά

Το επόμενο notebook θα είναι το `06_create_embeddings_and_vectorstore.ipynb`.